# Financial Risk Prediction — Credit Default Risk Modeling

End-to-end credit risk classification pipeline: EDA, feature engineering, handling class imbalance, model training (Logistic Regression, Random Forest, XGBoost, LightGBM), evaluation (ROC-AUC, KS statistic, Precision-Recall), and model explainability (SHAP).

## 1. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Data Loading

Load your credit risk dataset below. Expected columns include a binary target column (renamed to `target`) plus standard credit bureau features (utilization, age, debt ratio, monthly income, credit lines, delinquency history, dependents).

In [ ]:
df = pd.read_csv("your_dataset.csv", index_col=0)
df.rename(columns={"SeriousDlqin2yrs": "target"}, inplace=True)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Exploratory Data Analysis

In [ ]:
target_dist = df["target"].value_counts(normalize=True) * 100
target_dist

In [ ]:
fig, ax = plt.subplots()
df["target"].value_counts().plot(kind="bar", ax=ax, color=["#2E86AB", "#E63946"])
ax.set_xticklabels(["No Default", "Default"], rotation=0)
ax.set_title("Target Class Distribution")
ax.set_ylabel("Count")
plt.show()

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
missing

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.show()

In [ ]:
numeric_cols = [
    "RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio",
    "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans"
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for col, ax in zip(numeric_cols, axes.flatten()):
    sns.histplot(df[col].dropna(), bins=50, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Data Cleaning & Outlier Treatment

In [ ]:
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

df = df[df["age"] > 0]

late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]
for c in late_cols:
    df[c] = df[c].clip(upper=df[c].quantile(0.995))

df["DebtRatio"] = df["DebtRatio"].clip(upper=df["DebtRatio"].quantile(0.995))
df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(upper=2)

df.isnull().sum().sum()

## 5. Feature Engineering

In [ ]:
df["TotalPastDue"] = df[late_cols].sum(axis=1)
df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
df["CreditLinesPerAge"] = df["NumberOfOpenCreditLinesAndLoans"] / df["age"]
df["DebtToIncome"] = df["DebtRatio"] * df["MonthlyIncome"]
df["HasRealEstateLoan"] = (df["NumberRealEstateLoansOrLines"] > 0).astype(int)
df["UtilizationBucket"] = pd.cut(
    df["RevolvingUtilizationOfUnsecuredLines"],
    bins=[-0.01, 0.3, 0.6, 1.0, np.inf],
    labels=[0, 1, 2, 3]
).astype(int)

df.shape

## 6. Train / Test Split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## 7. Handling Class Imbalance (SMOTE)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

pd.Series(y_train_res).value_counts(normalize=True)

## 8. Model Training

### 8.1 Logistic Regression (Baseline)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_res, y_train_res)
log_reg_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, log_reg_probs)

### 8.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=20,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train_res, y_train_res)
rf_probs = rf.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, rf_probs)

### 8.3 XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, eval_metric="auc",
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(X_train_res, y_train_res)
xgb_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, xgb_probs)

### 8.4 LightGBM

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1
)
lgb_model.fit(X_train_res, y_train_res)
lgb_probs = lgb_model.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, lgb_probs)

## 9. Model Evaluation & Comparison

In [ ]:
models = {
    "Logistic Regression": log_reg_probs,
    "Random Forest": rf_probs,
    "XGBoost": xgb_probs,
    "LightGBM": lgb_probs
}

results = pd.DataFrame({
    "Model": list(models.keys()),
    "ROC-AUC": [roc_auc_score(y_test, p) for p in models.values()],
    "Average Precision": [average_precision_score(y_test, p) for p in models.values()]
}).sort_values("ROC-AUC", ascending=False)

results

In [ ]:
fig, ax = plt.subplots()
for name, probs in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for name, probs in models.items():
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ax.plot(recall, precision, label=name)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve Comparison")
ax.legend()
plt.show()

### 9.1 KS Statistic (Kolmogorov-Smirnov)

In [ ]:
def ks_statistic(y_true, y_prob):
    df_ks = pd.DataFrame({"y": y_true, "p": y_prob})
    df_ks = df_ks.sort_values("p", ascending=False)
    df_ks["cum_bad"] = (df_ks["y"] == 1).cumsum() / (df_ks["y"] == 1).sum()
    df_ks["cum_good"] = (df_ks["y"] == 0).cumsum() / (df_ks["y"] == 0).sum()
    return np.max(np.abs(df_ks["cum_bad"] - df_ks["cum_good"]))

for name, probs in models.items():
    print(f"{name}: KS = {ks_statistic(y_test, probs):.4f}")

### 9.2 Confusion Matrix — Best Model

In [ ]:
best_model_name = results.iloc[0]["Model"]
best_probs = models[best_model_name]
threshold = 0.5
preds = (best_probs >= threshold).astype(int)

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Default", "Default"],
            yticklabels=["No Default", "Default"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.show()

print(classification_report(y_test, preds))

## 10. Feature Importance

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots()
importances.plot(kind="barh", ax=ax, color="#2E86AB")
ax.invert_yaxis()
ax.set_title("XGBoost Feature Importance (Top 15)")
plt.show()

## 11. Model Explainability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=True)

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=X.columns, plot_type="bar", show=True)

## 12. Business Scorecard — Probability to Score Mapping

Converts predicted default probability into a standard credit score scale (300–850), following the industry-standard PDO (Points to Double the Odds) scaling used in banking scorecards.

In [ ]:
def probability_to_score(prob, base_score=600, base_odds=50, pdo=20):
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    odds = (1 - prob) / prob
    factor = pdo / np.log(2)
    offset = base_score - factor * np.log(base_odds)
    score = offset + factor * np.log(odds)
    return score.clip(300, 850)

credit_scores = probability_to_score(best_probs)

fig, ax = plt.subplots()
sns.histplot(credit_scores, bins=50, kde=True, ax=ax, color="#2E86AB")
ax.set_title("Predicted Credit Score Distribution")
ax.set_xlabel("Credit Score")
plt.show()

## 13. Save Final Model

In [ ]:
import joblib

joblib.dump(xgb_model, "credit_risk_xgb_model.pkl")
joblib.dump(scaler, "credit_risk_scaler.pkl")
print("Model and scaler saved.")

## 14. Summary

- Best performing model selected by ROC-AUC on held-out test set
- Class imbalance handled via SMOTE oversampling
- Explainability provided via SHAP values and feature importance
- Final output includes a probability-to-credit-score business scorecard